In [ ]:
suppressPackageStartupMessages({
    library(monocle3)
    library(MASS)
    library(reshape2)
    library(dplyr)
    library(ggplot2)
    library(grid)
    library(gridExtra)
    library(Matrix)
    library(IRdisplay)
    library(tibble)
    library(ggridges)
    library(pheatmap)
    library(stringr)
    library(scales)
    library(DelayedArray)
    library(hooke)
    library(splines)
    library(msigdbr)
    library(rmcorr)
    library(ComplexHeatmap)
    library(circlize)
    library(viridis)
    library(glmmTMB)
    library(ggeffects)
    library(ggridges)
    library(fgsea)
    library(clusterProfiler)
})

In [ ]:
pseudobulk_cds = function(cds_sub, sample_group = "Cluster_group", cell_group = "Grouping") {
    cf_ccs = new_cell_count_set(cds_sub,
                            sample_group = sample_group,
                            cell_group = cell_group)
    
    pb_cds = hooke:::pseudobulk_ccs_for_states(cf_ccs)
    
    return(pb_cds)
    
}

In [ ]:
aggregate_gene_score = function(cds, gene_list=c("CCNB1", "CCNB2","CDK1"), name = "gene_list_score"){
    cds_gene_list_norm = cds[rowData(cds)$gene_short_name %in% gene_list,] 
    aggregate_marker_expression = counts(cds_gene_list_norm)
    aggregate_marker_expression = t(t(aggregate_marker_expression) / colData(cds_gene_list_norm)$Size_Factor)
    aggregate_marker_expression = Matrix::colSums(aggregate_marker_expression)
    aggregate_marker_expression = log(aggregate_marker_expression + 1)
    colData(cds)[[name]] = aggregate_marker_expression
    return(cds)
}

normalize <- function(x) {
    return((x- min(x)) /(max(x)-min(x)))
}

detect_genes_fixed = function (cds, min_expr = 0) {
    assertthat::assert_that(methods::is(cds, "cell_data_set"))
    assertthat::assert_that(is.numeric(min_expr))
    mat_bin <- counts(cds) > min_expr
    rowData(cds)$num_cells_expressed <- Matrix::rowSums(mat_bin)
    colData(cds)$num_genes_expressed <- Matrix::colSums(mat_bin)
    cds
}


estimate_cell_cycle = function(cds, g1s_markers, g2m_markers){
    cds_g1s = cds[rowData(cds)$gene_short_name %in% g1s_markers,] 
    aggregate_g1s_expression = counts(cds_g1s)
    aggregate_g1s_expression = t(t(aggregate_g1s_expression) / colData(cds_g1s)$Size_Factor)
    aggregate_g1s_expression = Matrix::colSums(aggregate_g1s_expression)

    cds_g2m = cds[rowData(cds)$gene_short_name %in% g2m_markers,] 
    aggregate_g2m_expression = counts(cds_g2m)
    aggregate_g2m_expression = t(t(aggregate_g2m_expression) / colData(cds_g2m)$Size_Factor)
    aggregate_g2m_expression = Matrix::colSums(aggregate_g2m_expression)
    cds$g1s_score = log(aggregate_g1s_expression+1)
    cds$g2m_score = log(aggregate_g2m_expression+1)
    cds$proliferation_index = log(aggregate_g1s_expression + aggregate_g2m_expression + 1)
    return(cds)
}

In [ ]:
gene_set_BP = msigdbr::msigdbr(species = "mouse", subcategory = "GO:BP")
gene_set_CC = msigdbr::msigdbr(species = "mouse", subcategory = "GO:CC")
gene_set_hallmarks = msigdbr::msigdbr(species = "mouse", category = "H")
gene_set_reactome = msigdbr::msigdbr(species = "mouse", subcategory = "CP:REACTOME")
gene_set_KEGG = msigdbr::msigdbr(species = "mouse", subcategory = "CP:KEGG_LEGACY")

gene_set_BP_df = gene_set_BP %>% 
                dplyr::distinct(gs_name, gene_short_name=gene_symbol) %>% 
                as.data.frame()

gene_set_CC_df = gene_set_CC %>% 
                dplyr::distinct(gs_name, gene_short_name=gene_symbol) %>% 
                as.data.frame()

gene_set_hallmarks_df = gene_set_hallmarks %>% 
                dplyr::distinct(gs_name, gene_short_name=gene_symbol) %>% 
                as.data.frame()

gene_set_reactome_df = gene_set_reactome %>% 
                dplyr::distinct(gs_name, gene_short_name=gene_symbol) %>% 
                as.data.frame()

gene_set_KEGG_df = gene_set_KEGG %>% 
                dplyr::distinct(gs_name, gene_short_name=gene_symbol) %>% 
                as.data.frame()

## Load data

In [ ]:
res_df = readRDS("../data/res_gene_corr_df_intra_inter_joined.RDS")
stouffer_df = readRDS("../data/stouffer_df_nucleolar_fraction.RDS")
nucleolar_gene_df = readRDS("../data/nucleolar_gene_df.RDS")

pb_cds_nuc_frac_all = readRDS("../data/pb_cds_nucleolar_fraction_global.RDS")
pb_cds_nuc_frac = readRDS("../data/pb_cds_nucleolar_fraction_pseudotime.RDS")

In [ ]:
rowdata = as.data.frame(rowData(pb_cds_nuc_frac_all))
lineages_to_omit <- c("GABAergic", "Glutamatergic")

In [ ]:
cm_all = normalized_counts(pb_cds_nuc_frac_all)
expression_data <- t(as.matrix(cm_all[nucleolar_gene_df$gene_ID, , drop = FALSE]))
df_plot = as.data.frame(colData(pb_cds_nuc_frac_all)) %>%
                    cbind(., expression_data)

df_plot_long <- df_plot %>%
  tidyr::pivot_longer(
    cols = all_of(nucleolar_gene_df$gene_ID), 
    names_to = "Gene", 
    values_to = "Expression"
  ) %>%
  left_join(., rowdata %>% select(Gene = gene_ID, gene_short_name))

In [ ]:
## Supplementary Figure 17

options(repr.plot.width=18, repr.plot.height=18, repr.plot.res=300)
df_plot_long %>%
    ggplot(., aes(y = avg_score, x = Expression)) +
        geom_point(size = 0.3) + 
        facet_wrap(~gene_short_name, scales = "free") + 
        labs(x = "Expression", y = "Nucleolar fraction score") + 
        monocle3:::monocle_theme_opts() + 
        theme(axis.text.x = element_text(color = "black", size = 10)) + 
        theme(legend.position = "none", axis.title=element_text(size=20),
              strip.text = element_text(size = 16),
                      axis.text.y = element_text(size = 13, color = "black"), 
                      axis.ticks = element_line(color = "black"))


In [ ]:
## Supplementary Figure 18A

options(repr.plot.width=10, repr.plot.height=4, repr.plot.res=300)
res_df %>%
    filter(!trajectory_refined %in% lineages_to_omit) %>%
    filter(gene_ID %in% nucleolar_gene_df$gene_ID) %>%
    left_join(., nucleolar_gene_df %>%
                    select(gene_ID, global_cor = Spearman)) %>%
    mutate(gene_short_name = as.character(gene_short_name)) %>%

    ggplot(., aes(x = reorder(gene_short_name, Spearman, FUN = median, decreasing = T), 
                  y = Spearman)) +
        geom_boxplot(aes(fill = global_cor), fatten = 0.75, lwd = 0.75) + 
        labs(x = "", y = "Intra-lineage correlation (ρ)", fill = "Global\ncorrelation (ρ)") + 
        viridis::scale_fill_viridis(option = "viridis", limits = c(0.7, 0.87), begin = 0.2) + 
        monocle3:::monocle_theme_opts() + 
        theme(axis.text.x = element_text(angle = 45, vjust = 1, hjust=1, color = "black", size = 10)) + 
        theme(axis.title=element_text(size=13), 
              title = element_text(size = 10), 
              axis.text.y = element_text(color = "black", size = 11),
              axis.ticks = element_line(color = "black")) 

# Gene expression score heatmap (Figure 5A)

In [ ]:
traj_order = c("CNS_neurons", "Intermediate_progenitors", "Neuroectoderm_and_glia",
               "Epithelial", "Mesenchymal", "Hepatic", "Myogenic", 
               "Myeloid", "Erythroid")

celltype_to_keep = c("Mesoderm", "Muscle_cells", "Epithelium",
               "White_blood_cells", "Hepatocytes", "Definitive_erythroid")

metadata_plot_base = as.data.frame(colData(pb_cds_nuc_frac_all)) %>%
    filter(trajectory_refined %in% celltype_to_keep)


meta_df_ordered <- metadata_plot_base %>%
    filter(!trajectory_refined %in% c("CNS_neurons", "Intermediate_progenitors")) %>%
    mutate(trajectory_refined = case_match(trajectory_refined,
                                           "White_blood_cells" ~ "Myeloid",
                                           "Muscle_cells" ~ "Myogenic",
                                           "Epithelium" ~ "Epithelial",
                                           "Mesoderm" ~ "Mesenchymal",
                                           "Hepatocytes" ~ "Hepatic",
                                           "Definitive_erythroid" ~ "Erythroid",
                                            .default = trajectory_refined),
           trajectory_refined = gsub('_', ' ', trajectory_refined),
           trajectory_refined = factor(trajectory_refined, levels = gsub('_', ' ', traj_order))) %>%
    group_by(trajectory_refined) %>%
    arrange(avg_score, .by_group = TRUE) %>%
    mutate(local_score_gradient = (avg_score - min(avg_score)) / (max(avg_score) - min(avg_score))) %>%
    ungroup()

cell_ids <- meta_df_ordered$pseudobulk_id
cm <- normalized_counts(pb_cds_nuc_frac_all)
trajectories <- levels(meta_df_ordered$trajectory_refined)

In [ ]:
score_col_fun = colorRamp2(c(0, 1), c("#f0f0f0", "#555555"))
z_col_fun = colorRamp2(c(-2, 0, 2), c("#2166AC", "white", "#B2182B"))


my_new_label <- "Nucleolar fraction"

top_anno <- HeatmapAnnotation(
    nuc_frac = meta_df_ordered$local_score_gradient,

    col = list(nuc_frac = score_col_fun),
    
    annotation_label = " ", 
    annotation_name_rot = 0,

    annotation_name_gp = gpar(
        fontface = "bold", 
        fontsize = 13,
        lineheight = 0.9 
    ),
    
    height = unit(0.6, "cm"),
    
    show_annotation_name = TRUE,
    
    annotation_legend_param = list(
        nuc_frac = list(
            title = "Scaled\nnucleolar\nfraction", 
            direction = "vertical", 
            title_position = "topleft"
        )
    )
)


options(repr.plot.width=11.5, repr.plot.height=12, repr.plot.res=300)
grid.newpage() 

heatmap_matrix <- cm[as.character(nucleolar_gene_df$gene_ID), cell_ids]

local_scaled_list <- list()
for (traj in trajectories) {
    lineage_cells <- meta_df_ordered$pseudobulk_id[meta_df_ordered$trajectory_refined == traj]
    if(length(lineage_cells) == 0) next

    mat_subset <- heatmap_matrix[, lineage_cells, drop = FALSE]
    scaled_subset <- t(scale(t(mat_subset)))
    scaled_subset[is.na(scaled_subset)] <- 0 
    local_scaled_list[[as.character(traj)]] <- scaled_subset
}

heatmap_matrix_local_scaled <- do.call(cbind, local_scaled_list)
heatmap_matrix_local_scaled <- heatmap_matrix_local_scaled[, cell_ids]

gene_labels <- nucleolar_gene_df$gene_short_name[match(rownames(heatmap_matrix_local_scaled), nucleolar_gene_df$gene_ID)]
rownames(heatmap_matrix_local_scaled) <- gene_labels

ht <- Heatmap(
    heatmap_matrix_local_scaled,            
    col = z_col_fun,              
    name = "Scaled\nexpression",
    top_annotation = top_anno,
    column_split = meta_df_ordered$trajectory_refined, 
    column_gap = unit(3, "mm"),
    cluster_columns = FALSE,                
    cluster_rows = TRUE,                    
    show_column_names = FALSE,
    row_names_gp = gpar(fontsize = 15), 
    column_title_gp = gpar(fontsize = 15)
)

draw(ht, 
           heatmap_legend_side = "right", 
           annotation_legend_side = "right", 
           merge_legend = TRUE)

# Fit GLMM on scores

In [ ]:
nucleolar_genes <- nucleolar_gene_df$gene_short_name
col_name <- "gene_score_Z5_0.7_0.4_n_6"

pb_cds_nuc_frac_all <- aggregate_gene_score(pb_cds_nuc_frac_all, 
                                            nucleolar_genes, 
                                            col_name)

In [ ]:
metadata_fit = as.data.frame(colData(pb_cds_nuc_frac_all))
metadata_fit$norm_x = normalize(metadata_fit[[col_name]])
    
n <- nrow(metadata_fit)
metadata_fit$fraction_squeezed <- (normalize(metadata_fit$avg_score) * (n - 1) + 0.5) / n

model_glmm <- glmmTMB(
  fraction_squeezed ~ norm_x + (1 | Embryo), 
  data = metadata_fit, 
  family = beta_family(link = "cloglog")
)


pred_universal <- predict_response(
    model_glmm, 
    terms = "norm_x [0:1, by=0.01]",
    type = "fixed"
)

In [ ]:
## Figure 6B

options(repr.plot.width=4, repr.plot.height=4, repr.plot.res=300)

ggplot() +
  geom_point(data = metadata_fit, 
             aes(x = norm_x, y = fraction_squeezed), 
             alpha = 0.8, size = 0.7) +

  geom_line(data = pred_universal, 
            aes(x = x, y = predicted), 
            color = "red", linewidth = 0.8) +

  geom_ribbon(data = pred_universal, 
              aes(x = x, ymin = conf.low, ymax = conf.high), 
              fill = "red", alpha = 0.15) +

  labs(x = "Normalized geneset expression", 
       y = "Nucleolar fraction score",
       color = "Embryo") +
  monocle3:::monocle_theme_opts() +
  theme(legend.position = "right", aspect.ratio = 1) + 
    theme(
        strip.text.x = element_text(size = 13), 
        strip.text.y = element_text(size = 8.5, angle = 270),
        strip.background = element_blank(),
        
        axis.text = element_text(color="black", size = 15),
        axis.ticks = element_line(color = "black"),
        axis.title = element_text(color = "black", size = 15)
    )


In [ ]:
model_filename = "nucleolar_model_glmm_Z5_0.7_0.4_n_6.RDS"
saveRDS(model_glmm, model_filename)

In [ ]:
## Supplementary Figure 21A

options(repr.plot.width=8, repr.plot.height=8, repr.plot.res=300)

ggplot() +
  geom_point(data = metadata_fit, 
             aes(x = norm_x, y = fraction_squeezed), 
             alpha = 0.8, size = 0.7) +

  geom_line(data = pred_universal, 
            aes(x = x, y = predicted), 
            color = "red", linewidth = 0.8) +

  geom_ribbon(data = pred_universal, 
              aes(x = x, ymin = conf.low, ymax = conf.high), 
              fill = "red", alpha = 0.15) +

  facet_wrap(~Embryo, axes = "all") + 

  labs(x = "Normalized geneset expression", 
       y = "Nucleolar fraction score",
       color = "Embryo") +
  monocle3:::monocle_theme_opts() +
  theme(legend.position = "right", aspect.ratio = 1) + 
    theme(
        strip.text.x = element_text(size = 13), 
        strip.text.y = element_text(size = 8.5, angle = 270),
        strip.background = element_blank(),
        
        axis.text = element_text(color="black", size = 10),
        axis.ticks = element_line(color = "black"),
        axis.title = element_text(color = "black", size = 13)
    )


### Plot aggregate gene expression across lineages

In [ ]:
col_name = "nucleolar_gene_score"
pb_cds_nuc_frac = aggregate_gene_score(pb_cds_nuc_frac, 
                        nucleolar_genes, col_name)

In [ ]:
metadata_lineage = as.data.frame(colData(pb_cds_nuc_frac))

metadata_med_score = metadata_lineage %>%
                        group_by(trajectory_refined, sample) %>%
                        summarise(med_avg_score = median(avg_score), 
                                  med_gene_score = median(nucleolar_gene_score, na.rm = TRUE),
                                  .groups = "drop")

In [ ]:
rm_data <- metadata_lineage %>%
    mutate(trajectory_refined = case_match(trajectory_refined,
                                           "White_blood_cells" ~ "Myeloid",
                                           "Muscle_cells" ~ "Myogenic",
                                           "Epithelium" ~ "Epithelial",
                                           "Fibroblasts_Fap+" ~ "Fibroblasts",
                                           "Hepatocytes" ~ "Hepatic",
                                           "Definitive_erythroid" ~ "Erythroid",
                                           "Upper_layer_neurons" ~ "Upper layer\nneurons",
                                           "GABAergic" ~ "GABAergic\nneurons",
                                           "Glutamatergic" ~ "Glutamatergic\nneurons",
                                           .default = trajectory_refined)) %>%
    mutate(Embryo = factor(Embryo))

In [ ]:
rm_ranked_results = rm_data %>%
    filter(!is.na(avg_score), !is.na(nucleolar_gene_score)) %>%
    group_by(trajectory_refined) %>%
    group_modify(~ {
        valid_embryos <- .x %>%
            count(Embryo) %>%
            filter(n >= 3) %>%
            pull(Embryo)

        df_sub <- .x %>% filter(Embryo %in% valid_embryos)

        if (length(unique(df_sub$Embryo)) < 2) {
            return(data.frame(r_rm = NA, p_val = NA, n_embryos = length(unique(df_sub$Embryo))))
        }

        tryCatch({
            res <- rmcorr(participant = Embryo, 
                          measure1 = rank(avg_score), 
                          measure2 = rank(nucleolar_gene_score), 
                          dataset = df_sub)

            data.frame(
                r_rm = round(res$r, 3),
                p_val = res$p,
                n_embryos = length(unique(df_sub$Embryo))
            )
        }, error = function(e) {
            data.frame(r_rm = NA, p_val = NA, n_embryos = length(unique(df_sub$Embryo)))
        })
    }) %>%
    ungroup()

rm_ranked_results <- rm_ranked_results %>%
    mutate(label = ifelse(is.na(r_rm), 
                          'Insuff. Data',
                          paste0("ρ = ", r_rm)))

rm_ranked_results

In [ ]:
## Supplementary Figure 18B

label_pos <- rm_data %>%
    group_by(trajectory_refined) %>%
    summarise(
        x_pos = min(avg_score, na.rm = TRUE),
        y_pos = max(nucleolar_gene_score, na.rm = TRUE),
        .groups = "drop"
    ) %>%
    left_join(rm_ranked_results, by = "trajectory_refined")


options(repr.plot.width=10, repr.plot.height=9, repr.plot.res=300)

ggplot(rm_data, aes(x = avg_score, y = nucleolar_gene_score)) +
    geom_point(size = 0.8, aes(color = Embryo), alpha = 0.8) +

    geom_label(data = filter(label_pos, !is.na(label)), 
              aes(x = x_pos, y = y_pos, label = label), 
              x = -Inf, y = Inf,    
              hjust = -0.1, vjust = 1.1,
              size = 4, 
              fill = "white",       
              label.size = NA,      
              label.padding = unit(0.2, "lines"), 
              parse = F) +
    
    facet_wrap(~trajectory_refined, scales = "free") +
    labs(x = "Nucleolar fraction score", 
         y = "Aggregated geneset expression") +
    monocle3:::monocle_theme_opts() + 
    theme(aspect.ratio = 1,
        strip.text.x = element_text(size = 13), 
        strip.text.y = element_text(size = 8.5, angle = 270),
        strip.background = element_blank(),
        
        axis.text = element_text(color="black", size = 10),
        axis.ticks = element_line(color = "black"),
        axis.title = element_text(color = "black", size = 13)
    ) + 
        guides(color = guide_legend(override.aes = list(size = 4)))

## Correlation against known genesets

In [ ]:
gene_set_of_interest = rbind(gene_set_hallmarks_df,
                             gene_set_CC_df,
                             gene_set_KEGG_df)
unique_genesets <- unique(gene_set_of_interest$gs_name)
length(unique_genesets)

for (gs in unique_genesets) {
    genes_in_set <- gene_set_of_interest %>% 
    filter(gs_name == gs) %>% 
    pull(gene_short_name)

    pb_cds_nuc_frac <- aggregate_gene_score(pb_cds_nuc_frac, 
                                          gene_list = genes_in_set, 
                                          name = gs)
}

In [ ]:
run_combined_rmcorr <- function(gene_expr, id, score) {
  df_pearson <- data.frame(Embryo = id, Score = score, Expr = gene_expr)
  df_spearman <- data.frame(Embryo = id, 
                            Score_Ranked = rank(score, ties.method = "average"), 
                            Expr_Ranked = rank(gene_expr, ties.method = "average"))
  
  spearman_res <- tryCatch({
    rmc_s <- rmcorr(participant = Embryo, measure1 = Score_Ranked, measure2 = Expr_Ranked, dataset = df_spearman)
    c(Spearman = rmc_s$r, Spearman_pvalue = rmc_s$p)
  }, error = function(e) c(Spearman = NA, Spearman_pvalue = NA))
  
  pearson_res <- tryCatch({
    rmc_p <- rmcorr(participant = Embryo, measure1 = Score, measure2 = Expr, dataset = df_pearson)
    c(Pearson = rmc_p$r, Pearson_pvalue  = rmc_p$p)
  }, error = function(e) c(Pearson = NA, Pearson_pvalue = NA))
  
  return(c(spearman_res, pearson_res))
}

In [ ]:
res_df_gs <- NULL

for (traj in unique(pb_cds_nuc_frac$trajectory_refined)) {

    meta_sub <- as.data.frame(colData(pb_cds_nuc_frac)) %>%
    filter(trajectory_refined == traj)

    subject_id <- as.factor(meta_sub$Embryo) 
    score_vec <- meta_sub$avg_score


    traj_res_list <- list()

    for (gs in unique_genesets) {
        gs_expr <- meta_sub[[gs]]

        if (sum(gs_expr, na.rm = TRUE) == 0) {
            traj_res_list[[gs]] <- c(Spearman = NA, Spearman_pvalue = NA, Pearson = NA, Pearson_pvalue = NA)
            next
        }

        res <- run_combined_rmcorr(gene_expr = gs_expr, id = subject_id, score = score_vec)
        traj_res_list[[gs]] <- res
    }

    gs_cor_df_tmp <- do.call(rbind, traj_res_list) %>%
    as.data.frame() %>%
    rownames_to_column("gs_name") %>%
    mutate(trajectory_refined = traj,
           Spearman_FDR = p.adjust(Spearman_pvalue, method = "BH"),
           Pearson_FDR = p.adjust(Pearson_pvalue, method = "BH")) 

    if (is.null(res_df_gs)) {
        res_df_gs <- gs_cor_df_tmp
    } else {
        res_df_gs <- rbind(res_df_gs, gs_cor_df_tmp)
    }

    print(paste0(traj, ': Processed ', length(unique_genesets), ' genesets'))
    flush.console()
}

In [ ]:
stouffer_gs_df <- res_df_gs %>%
    mutate(Spearman_pvalue = ifelse(Spearman_pvalue == 0, 1e-300, Spearman_pvalue)) %>%

    mutate(
        Z_score = sign(Spearman) * qnorm(Spearman_pvalue / 2, lower.tail = FALSE)
    ) %>%

    group_by(gs_name) %>%
    summarize(
        mean_Spearman = mean(Spearman, na.rm = TRUE),
        median_Spearman = median(Spearman, na.rm = TRUE),
        k_trajectories = n(),                       
        sum_Z = sum(Z_score, na.rm = TRUE),          
        n_positive_0.2 = sum(Spearman > 0.2, na.rm = TRUE), 
        n_positive_0.3 = sum(Spearman > 0.3, na.rm = TRUE), 
        n_positive_0.4 = sum(Spearman > 0.4, na.rm = TRUE), 
        n_negative = sum(Spearman < -0.3, na.rm = TRUE),   
        .groups = "drop"
    ) %>%

    mutate(
        Stouffer_Z = sum_Z / sqrt(k_trajectories),    
        Meta_Pvalue = 2 * pnorm(-abs(Stouffer_Z)),         
        Meta_FDR = p.adjust(Meta_Pvalue, method = "BH")    
    ) %>%

    arrange(desc(Stouffer_Z))

head(stouffer_gs_df, 5)

In [ ]:
stouffer_gs_df %>%
    filter(Meta_FDR < 0.01) %>% 
    filter(Stouffer_Z > 8 & n_positive_0.3 > 6 & n_negative < 3 & median_Spearman > 0.4) %>%
    filter(!grepl("REACTOME", gs_name)) %>%
    slice(1:30) %>% pull(gs_name) %>%
    print()

In [ ]:
theme_mapping <- list(
    "Growth &\nmetabolism" = c( 
        "HALLMARK_MYC_TARGETS_V1", "HALLMARK_MYC_TARGETS_V2", 
        "KEGG_PURINE_METABOLISM", "KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM", 
        "KEGG_PROPANOATE_METABOLISM", "KEGG_GLYCOSAMINOGLYCAN_BIOSYNTHESIS_CHONDROITIN_SULFATE", 
        "KEGG_CYSTEINE_AND_METHIONINE_METABOLISM", 
        "KEGG_SPHINGOLIPID_METABOLISM",     
        "GOCC_POLYPRENYL_DIPHOSPHATE_SYNTHASE_COMPLEX"
    ),

    "Ribosome\nbiogenesis" = c( 
        "GOCC_PRERIBOSOME_LARGE_SUBUNIT_PRECURSOR", "GOCC_PRERIBOSOME", "GOCC_90S_PRERIBOSOME", 
        "GOCC_RNAI_EFFECTOR_COMPLEX", "GOCC_SNO_S_RNA_CONTAINING_RIBONUCLEOPROTEIN_COMPLEX", 
        "GOCC_CAJAL_BODY", "GOCC_BOX_C_D_RNP_COMPLEX", "GOCC_SM_LIKE_PROTEIN_FAMILY_COMPLEX",
        "GOCC_NUCLEAR_EXOSOME_RNASE_COMPLEX", 
        "GOCC_CYTOPLASMIC_EXOSOME_RNASE_COMPLEX",
        "GOCC_SMALL_SUBUNIT_PROCESSOME",
        "GOCC_EXORIBONUCLEASE_COMPLEX"  
    ),

    "Protein\nsynthesis" = c( 
        "KEGG_AMINOACYL_TRNA_BIOSYNTHESIS", 
        "GOCC_EUKARYOTIC_TRANSLATION_INITIATION_FACTOR_4F_COMPLEX", 
        "GOCC_NATA_COMPLEX"  
    ),

    "Proteostasis" = c( 
        "GOCC_CYTOSOLIC_PROTEASOME_COMPLEX", 
        "GOCC_PEPTIDASE_COMPLEX"
    ),

    "Intracellular\ntrafficking" = c( 
        "GOCC_NUCLEAR_PORE_OUTER_RING", 
        "GOCC_NUCLEAR_OUTER_MEMBRANE", 
        "GOCC_NUCLEAR_OUTER_MEMBRANE_ENDOPLASMIC_RETICULUM_MEMBRANE_NETWORK", 
        "GOCC_CYTOPLASMIC_STRESS_GRANULE"
    )
)

theme_df <- data.frame(
    gs_name = unlist(theme_mapping),
    Theme = rep(names(theme_mapping), lengths(theme_mapping))
)

In [ ]:
theme_lineage_dist <- res_df_gs %>%
    filter(gs_name %in% theme_df$gs_name) %>%
    left_join(theme_df, by = "gs_name") %>%
    mutate(
        Theme = gsub("_", " ", Theme),
        is_sig = ifelse(Spearman_pvalue < 0.05, "Significant", "Non-significant")
    )

theme_lineage_dist <- theme_lineage_dist %>%
    mutate(lineage_supergroup = case_when(
        trajectory_refined %in% c("Astrocytes", "GABAergic", "Glutamatergic", "Upper_layer_neurons") ~ "Neural",
        trajectory_refined %in% c("Chondrocytes", "Fibroblasts_Fap+", "Osteoblasts") ~ "Mesenchymal",
        trajectory_refined == "White_blood_cells" ~ "Myeloid",
        trajectory_refined == "Muscle_cells"      ~ "Myogenic",
        trajectory_refined == "Epithelium"        ~ "Epithelial",
        trajectory_refined == "Hepatocytes"       ~ "Hepatic",
        trajectory_refined == "Definitive_erythroid" ~ "Erythroid",
        .default = trajectory_refined
    ))

In [ ]:
## Figure 5B

options(repr.plot.width=8.5, repr.plot.height=4.5, repr.plot.res=300)
ggplot(theme_lineage_dist, aes(y = reorder(Theme, Spearman, FUN = median, decreasing = F), 
                               x = Spearman)) +
    geom_hline(yintercept = 0, linetype = "dashed", color = "gray60", size = 0.8) +
    geom_violin(fill = "gray90", color = NA, alpha = 1, trim = FALSE) +
    geom_boxplot(width = 0.2, outlier.shape = NA, alpha = 0.5, color = "black", lwd = 0.9, fatten = 0.9) +

    geom_jitter(aes(alpha = is_sig, color = lineage_supergroup),
              width = 0.05, size = 1.3) +

    scale_alpha_manual(values = c("Significant" = 0.8, "Non-significant" = 0.2), guide = "none") +

    coord_cartesian(xlim = c(-1, 1.2)) +
    labs(y = "",
       x = "Spearman Correlation (ρ)",
       color = "Lineage") +
    theme_minimal() +
        scale_color_viridis_d(option = "turbo") + 
        theme(axis.line = element_line(color = "black"),
        axis.text = element_text(size = 14, color = "black"),
        axis.title = element_text(size = 14, color = "black"),
        legend.position = "right",
        legend.text = element_text(size = 12, color = "black"),
        legend.spacing.y = unit(0.4, 'cm'),
        legend.title = element_text(size = 14),
    ) +
    guides(color = guide_legend(override.aes = list(size = 5, alpha = 1)))

In [ ]:
## Supplementary Figure 19A

options(repr.plot.width=9, repr.plot.height=6.5, repr.plot.res=300)
ggplot(theme_lineage_dist, aes(x = Spearman, y = lineage_supergroup, fill = Theme)) +
    geom_vline(xintercept = 0, linetype = "dashed", alpha = 0.4) +
    stat_binline(aes(height = after_stat(ndensity)),
                 bins = 20, 
                 alpha = 0.7, 
                 scale = 0.8, 
                 draw_baseline = FALSE, 
                 color = "black") +
    facet_wrap(~Theme, ncol = 3, axes = "all_x") +
    labs(x = "Spearman Correlation (ρ)", y = "") +
    scale_y_discrete(expand = expansion(mult = c(0.05, 0.15))) +
    theme_minimal() + 
    theme(panel.grid.minor.y = element_line(color = "grey92", linewidth = 0.2),
        axis.title.x = element_text(hjust = 0.5, size = 13, face = "bold", margin = margin(t = 10)),
        axis.title.y = element_text(hjust = 0.5, size = 13, face = "bold", margin = margin(r = 10)),
        axis.text.x = element_text(size = 10, color = "black"),
        
        strip.text = element_text(size = 14, face = "bold", margin = margin(t = 12, b = 12)),
        legend.position = "none",
        strip.background = element_blank(),
        panel.spacing = unit(1.5, "lines")
    )

In [ ]:
genesets = gene_set_hallmarks_df

m_list <- split(x = genesets$gene_short_name, 
                f = genesets$gs_name)

m_list_upper <- lapply(m_list, str_to_upper)

In [ ]:
stouffer_clean <- stouffer_df %>%
    filter(!is.na(gene_short_name), gene_short_name != "") %>%
    mutate(gene_name_upper = toupper(gene_short_name)) %>%
    group_by(gene_name_upper) %>%
    summarize(Stouffer_Z = mean(Stouffer_Z, na.rm = TRUE), .groups = "drop")


ranks <- stouffer_clean$Stouffer_Z
names(ranks) <- stouffer_clean$gene_name_upper

ranks_final <- ranks - mean(ranks, na.rm = TRUE)
ranks_final <- ranks_final + rnorm(length(ranks_final), 0, 1e-9)
ranks_final <- sort(ranks_final, decreasing = TRUE)

print(paste("Unique genes for GSEA:", length(ranks_final)))
print(paste("Matching genes in Hallmark:", length(intersect(names(ranks_final), unlist(m_list_upper)))))

In [ ]:
## fgsea on stouffer score on Hallmark
gsea_results_hallmark <- fgsea(pathways = m_list_upper, 
                      stats = ranks_final, 
                      minSize = 10, 
                      maxSize = 500,
                      scoreType = "pos")

gsea_results_hallmark <- gsea_results_hallmark %>% 
                    filter(padj < 0.01) %>%
                    arrange(desc(NES))

head(gsea_results_hallmark, 10)

In [ ]:
## fgsea on stouffer score on GO:CC

genesets = gene_set_CC_df

m_list <- split(x = genesets$gene_short_name, 
                f = genesets$gs_name)

m_list_upper <- lapply(m_list, str_to_upper)

gsea_results_CC <- fgsea(pathways = m_list_upper, 
                      stats = ranks_final, 
                      minSize = 10, 
                      maxSize = 500,
                      scoreType = "pos")

gsea_results_CC <- gsea_results_CC %>% 
                    filter(padj < 0.01) %>%
                    arrange(desc(NES))

head(gsea_results_CC, 20)

In [ ]:
enriched_genes_df <- gsea_results_hallmark %>%
    filter(padj < 0.05) %>%  
    select(pathway, NES, padj, leadingEdge) %>%
    tidyr::unnest(leadingEdge) %>%  
    rename(gene_symbol = leadingEdge)

enriched_genes_df <- enriched_genes_df %>%
    left_join(stouffer_clean, by = c("gene_symbol" = "gene_name_upper"))

head(enriched_genes_df)

In [ ]:
core_genes <- enriched_genes_df %>%
    group_by(gene_symbol, Stouffer_Z) %>%
    summarize(n_pathways = n(), 
            pathway_list = paste(pathway, collapse = ", "),
            .groups = "drop") %>%
arrange(desc(n_pathways), desc(Stouffer_Z))

head(core_genes, 20)

In [ ]:
h_top7 <- gsea_results_hallmark %>% 
    arrange(desc(NES)) %>% 
    slice_head(n = 7) %>% 
    mutate(Source = "Hallmark")

cc_top7 <- gsea_results_CC %>% 
    arrange(desc(NES)) %>% 
    slice_head(n = 7) %>% 
    mutate(Source = "GO Cellular Component")

top_hits <- rbind(h_top7, cc_top7) %>%
    mutate(pathway = str_remove_all(pathway, "HALLMARK_|GOCC_")) %>%
    mutate(pathway = str_replace_all(pathway, "_", " ")) %>%
    mutate(pathway = str_wrap(pathway, width = 20)) %>%
    mutate(pathway = reorder(pathway, NES))

In [ ]:
## Supplementary Figure 19B

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=300)

ggplot(top_hits, aes(x = pathway, y = NES, fill = Source)) +
    geom_col(alpha = 0.8, color = "black", linewidth = 0.2) +
    facet_wrap(~Source, scales = "free", ncol = 2) + 
    coord_flip() +

    scale_fill_manual(values = c("Hallmark" = "#C06C6C", "GO Cellular Component" = "#4A74A4")) +
    labs(x = "", y = "Normalized Enrichment Score (NES)") +
    theme_minimal() +
    theme(
        legend.position = "none",
        strip.text = element_text(size = 12, face = "bold"),
        axis.text.x = element_text(size = 14, color = "black"),
        axis.text.y = element_text(size = 12, color = "black"),
        axis.title = element_text(size = 14, color = "black"),
        panel.grid.minor = element_blank(),
        plot.title = element_text(size = 15, face = "bold")
    )

### clusterProfiler on Hallmark and GO:CC

In [ ]:
my_hits <- toupper(nucleolar_genes)
my_universe <- toupper(stouffer_df$gene_short_name)

hallmark_t2g <- gene_set_hallmarks_df %>% 
    select(gs_name, gene_short_name) %>% 
    mutate(gene_short_name = toupper(gene_short_name))

In [ ]:
ora_results_hallmark <- enricher(
    gene = my_hits,
    universe = my_universe,
    pAdjustMethod = "BH",
    pvalueCutoff = 0.05,
    qvalueCutoff = 0.05,
    TERM2GENE = hallmark_t2g
)

ora_df <- as.data.frame(ora_results_hallmark)
head(ora_df)

ora_results_cc <- enricher(
    gene = my_hits,
    universe = my_universe,
    pAdjustMethod = "BH",
    pvalueCutoff = 0.05,
    qvalueCutoff = 0.05,
    TERM2GENE = gene_set_CC_df %>% 
    select(gs_name, gene_short_name) %>% 
        mutate(gene_short_name = toupper(gene_short_name))
)

ora_df <- as.data.frame(ora_results_cc)
head(ora_df)

In [ ]:
df_hallmark <- as.data.frame(ora_results_hallmark) %>% mutate(Source = "Hallmark")
df_cc <- as.data.frame(ora_results_cc) %>% mutate(Source = "Cell Component")

combined_df <- bind_rows(df_hallmark, df_cc) %>%
    mutate(GeneRatio_numeric = sapply(GeneRatio, function(x) {
        parts <- as.numeric(unlist(strsplit(x, "/")))
        parts[1] / parts[2]
    })) %>%
    mutate(Description = stringr::str_replace_all(Description, "_", " ")) %>%
    mutate(Description = stringr::str_wrap(Description, width = 17)) %>%
    mutate(Description = str_replace(Description, 
                                   "OLIGOSACCHARYLTRANSFERASE", 
                                   "OLIGOSACCHARYL-\nTRANSFERASE")) %>%

    arrange(desc(GeneRatio_numeric)) %>%
    mutate(Description = factor(Description, levels = rev(unique(Description))))

plot_data <- combined_df %>%
    group_by(Source) %>%
    slice_max(order_by = GeneRatio_numeric, n = 15) %>%
    ungroup()

In [ ]:
## Supplementary Figure 19C

options(repr.plot.width=5, repr.plot.height=5, repr.plot.res=300)

ggplot(plot_data, aes(x = GeneRatio_numeric, y = Description)) +
    geom_point(aes(size = Count, fill = p.adjust), 
             shape = 21, 
             color = "black", 
             stroke = 0.6) + 

    scale_fill_gradientn(
        colors = c("#4A74A4", "#8A7C92", "#C06C6C"),
        trans = "log10", 
        guide = guide_colorbar(
          reverse = TRUE, 
          ticks.colour = "black",
          barwidth = unit(3.5, "cm"),  
          barheight = unit(0.5, "cm"), 
          title.position = "top"
        )
    ) +

    scale_size_continuous(
        range = c(2, 6),
        guide = guide_legend(
          override.aes = list(fill = "white"),
          title.position = "top",
          keywidth = unit(0.1, "cm"),
          keyheight = unit(0.1, "cm")
        )
    ) +

    labs(x = "", y = "", fill = "Adjusted p-value") +
    theme_minimal() +
    theme(
        legend.position = "top",
        panel.grid.minor = element_blank(),
        axis.text.y = element_text(size = 8.5, lineheight = 0.8),
        axis.text.x = element_text(size = 12),
        strip.text = element_text(size = 12, face = "bold")
    )

## Negative genes

In [ ]:
stouffer_df %>%
    filter(Stouffer_Z < -5 & n_negative > 5 & n_positive_0.3 < 2 & median_Spearman < -0.4) %>%
    arrange(Stouffer_Z) %>%
    pull(gene_short_name) %>% print()

In [ ]:
negative_genes = stouffer_df %>%
    filter(Stouffer_Z < -5 & n_negative > 5 & n_positive_0.3 < 2 & median_Spearman < -0.4) %>%
    arrange(Stouffer_Z)
negative_genes

## Expression heatmap (Supplementary Figure 20A)

In [ ]:
traj_order = c("CNS neurons", "Intermediate & \nneuronal progenitors", "Neuroectoderm\n& glia",
               "Epithelial", "Mesenchymal", "Myogenic", 
               "Myeloid", "Hepatic", "Erythroid")

celltype_to_keep = c("CNS_neurons", "Intermediate_progenitors", "Brain_and_spinal_cord", 
                     "Mesoderm", "Muscle_cells", "Epithelium",
               "White_blood_cells", "Hepatocytes", "Definitive_erythroid")

metadata_plot_base = as.data.frame(colData(pb_cds_nuc_frac_all)) %>%
    filter(trajectory_refined %in% celltype_to_keep)


meta_df_ordered <- metadata_plot_base %>%
    mutate(trajectory_refined = case_match(trajectory_refined,
                           "CNS_neurons" ~ "CNS neurons", 
                           "Brain_and_spinal_cord" ~ "Neuroectoderm\n& glia",
                           "Intermediate_progenitors" ~ "Intermediate & \nneuronal progenitors",
                           "White_blood_cells" ~ "Myeloid",
                           "Muscle_cells" ~ "Myogenic",
                           "Epithelium" ~ "Epithelial",
                           "Mesoderm" ~ "Mesenchymal",
                           "Hepatocytes" ~ "Hepatic",
                           "Definitive_erythroid" ~ "Erythroid",
                            .default = trajectory_refined),
           trajectory_refined = gsub('_', ' ', trajectory_refined),
           trajectory_refined = factor(trajectory_refined, levels = gsub('_', ' ', traj_order))) %>%
    group_by(trajectory_refined) %>%
    arrange(avg_score, .by_group = TRUE) %>%
    mutate(local_score_gradient = (avg_score - min(avg_score)) / (max(avg_score) - min(avg_score))) %>%
    ungroup()


cell_ids <- meta_df_ordered$pseudobulk_id
cm <- normalized_counts(pb_cds_nuc_frac_all)
trajectories <- levels(meta_df_ordered$trajectory_refined)

In [ ]:
score_col_fun = colorRamp2(c(0, 1), c("#f0f0f0", "#555555"))
z_col_fun = colorRamp2(c(-2, 0, 2), c("#2166AC", "white", "#B2182B"))


my_new_label <- "Nucleolar fraction"

top_anno <- HeatmapAnnotation(
    nuc_frac = meta_df_ordered$local_score_gradient,

    col = list(nuc_frac = score_col_fun),
    
    annotation_label = " ", 
    annotation_name_rot = 0,
    
    annotation_name_gp = gpar(
        fontface = "bold", 
        fontsize = 13,
        lineheight = 0.9 
    ),
    
    height = unit(0.6, "cm"),
    
    show_annotation_name = TRUE,
    
    annotation_legend_param = list(
        nuc_frac = list(
            title = "Scaled\nnucleolar\nfraction", 
            direction = "vertical", 
            title_position = "topleft"
        )
    )
)


options(repr.plot.width=16, repr.plot.height=5.5, repr.plot.res=300)
grid.newpage() 

heatmap_matrix <- cm[negative_genes$gene_ID, cell_ids]

local_scaled_list <- list()
for (traj in trajectories) {
    lineage_cells <- meta_df_ordered$pseudobulk_id[meta_df_ordered$trajectory_refined == traj]
    if(length(lineage_cells) == 0) next

    mat_subset <- heatmap_matrix[, lineage_cells, drop = FALSE]
    scaled_subset <- t(scale(t(mat_subset)))
    scaled_subset[is.na(scaled_subset)] <- 0 
    local_scaled_list[[as.character(traj)]] <- scaled_subset
}

heatmap_matrix_local_scaled <- do.call(cbind, local_scaled_list)
heatmap_matrix_local_scaled <- heatmap_matrix_local_scaled[, cell_ids]

gene_labels <- negative_genes$gene_short_name[match(rownames(heatmap_matrix_local_scaled), negative_genes$gene_ID)]
rownames(heatmap_matrix_local_scaled) <- gene_labels


ht <- Heatmap(
    heatmap_matrix_local_scaled,            
    col = z_col_fun,              
    name = "Scaled\nexpression",
    top_annotation = top_anno,
    column_split = meta_df_ordered$trajectory_refined, 
    column_gap = unit(3, "mm"),
    cluster_columns = FALSE,                
    cluster_rows = TRUE,                    
    show_column_names = FALSE,
    row_names_gp = gpar(fontsize = 13), 
    column_title_gp = gpar(fontsize = 12)
)

draw(ht, 
       heatmap_legend_side = "right", 
       annotation_legend_side = "right", 
       merge_legend = TRUE)

In [ ]:
my_hits = toupper(negative_genes$gene_short_name)
my_universe <- toupper(stouffer_df$gene_short_name)

hallmark_t2g <- rbind(gene_set_hallmarks_df, gene_set_BP_df,
                             gene_set_CC_df,
                             gene_set_KEGG_df,
                             gene_set_reactome_df) %>% 
    select(gs_name, gene_short_name) %>% 
    mutate(gene_short_name = toupper(gene_short_name))

In [ ]:
ora_results <- enricher(
    gene = my_hits,
    universe = my_universe,
    pAdjustMethod = "BH",
    pvalueCutoff = 0.05,
    qvalueCutoff = 0.05,
    TERM2GENE = hallmark_t2g
)

ora_df <- as.data.frame(ora_results)
head(ora_df)

In [ ]:
## Supplementary Figure 20B

ora_results@result <- ora_results@result %>%
  mutate(GeneRatio_numeric = sapply(GeneRatio, function(x) {
    parts <- as.numeric(unlist(strsplit(x, "/")))
    parts[1] / parts[2]
  })) %>%
  arrange(desc(GeneRatio_numeric)) %>%
  mutate(Description = factor(Description, levels = rev(unique(Description))))

options(repr.plot.width=5.5, repr.plot.height=5.5, repr.plot.res=300)

enrichplot::dotplot(ora_results, showCategory = 15) + 
    labs(x = "", y = "", fill = "Adjusted p-value") +
    theme_minimal() +
    theme(legend.position = "top",
          panel.grid.minor = element_blank(),
          axis.text.x = element_text(size = 10),
          axis.text.y = element_text(size = 9),
          plot.title = element_text(size = 14, face = "bold")
    ) 

## Supplementary figure 20C

In [ ]:
negative_genes = stouffer_df %>%
    filter(Stouffer_Z < -5 & n_negative > 7 & n_positive_0.3 < 2 & median_Spearman < -0.4) %>%
    arrange(Stouffer_Z)
negative_genes$gene_short_name %>% print

In [ ]:
genes_to_plot <- negative_genes$gene_short_name

cds_subset <- pb_cds_nuc_frac[rowData(pb_cds_nuc_frac)$gene_short_name %in% genes_to_plot, ]
expr_matrix <- as.matrix(normalized_counts(cds_subset)) 
rownames(expr_matrix) <- rowData(cds_subset)$gene_short_name

expr_long <- as.data.frame(t(expr_matrix)) %>%
  tibble::rownames_to_column("pseudobulk_id") %>%
  tidyr::pivot_longer(cols = -pseudobulk_id, names_to = "gene", values_to = "expression")

metadata_df <- as.data.frame(colData(pb_cds_nuc_frac)) %>%
    mutate(Embryo = factor(Embryo)) %>%
    mutate(trajectory_refined = case_when(
        trajectory_refined == "Glutamatergic" ~ "Glutamatergic neurons",
        trajectory_refined == "GABAergic" ~ "GABAergic neurons",
        trajectory_refined == "Upper_layer_neurons" ~ "Upper layer neurons",
        trajectory_refined == "Fibroblasts_Fap+" ~ "Fibroblasts",
        trajectory_refined == "White_blood_cells" ~ "Myeloid",
        trajectory_refined == "Muscle_cells"      ~ "Myogenic",
        trajectory_refined == "Epithelium"        ~ "Epithelial",
        trajectory_refined == "Hepatocytes"       ~ "Hepatic",
        trajectory_refined == "Definitive_erythroid" ~ "Erythroid",
        .default = trajectory_refined
    ))

plot_data <- metadata_df %>%
  left_join(expr_long, by = "pseudobulk_id")

In [ ]:
options(repr.plot.width=11, repr.plot.height=8, repr.plot.res=300)

for (neg_gene in negative_genes$gene_short_name) {
    gene_data <- plot_data %>% 
        filter(gene == neg_gene)

    rm_ranked_results <- gene_data %>%
        filter(!is.na(avg_score), !is.na(expression)) %>%
        group_by(trajectory_refined) %>%
        group_modify(~ {
          valid_embryos <- .x %>%
            count(Embryo) %>%
            filter(n >= 3) %>%
            pull(Embryo)

          df_sub <- .x %>% filter(Embryo %in% valid_embryos)

          if (length(unique(df_sub$Embryo)) < 2) {
            return(data.frame(r_rm = NA, p_val = NA, n_embryos = length(unique(df_sub$Embryo))))
          }

          tryCatch({
            res <- rmcorr(participant = Embryo, 
                          measure1 = rank(avg_score), 
                          measure2 = rank(expression), 
                          dataset = df_sub)

            data.frame(
              r_rm = round(res$r, 3),
              p_val = res$p,
              n_embryos = length(unique(df_sub$Embryo))
            )
          }, error = function(e) {
            data.frame(r_rm = NA, p_val = NA, n_embryos = length(unique(df_sub$Embryo)))
          })
        }) %>%
        ungroup() %>%
        mutate(
          label = ifelse(is.na(r_rm), NA, 
                         paste0("ρ = ", r_rm))
        )

    p <- ggplot(gene_data, aes(x = avg_score, y = expression)) +

            geom_point(aes(color = Embryo), size = 1, alpha = 0.6) +

            facet_wrap(~trajectory_refined, scales = "free", ncol = 4) + 

            geom_label(data = filter(rm_ranked_results, !is.na(label)), 
                       aes(x = -Inf, y = -Inf, label = label), 
                       hjust = -0.3, vjust = -0.2, # vjust < 0 pushes it UP from the bottom edge
                       size = 3.6, 
                       color = "black",
                       fill = "white",             
                       label.padding = unit(0.2, "lines"), 
                       label.size = NA) +

            labs(x = "Nucleolar score", 
                 y = paste0(neg_gene, " expression")) +

            monocle3:::monocle_theme_opts() + 
            theme(
              plot.title = element_text(size = 16, face = "bold.italic", hjust = 0.5),
              strip.text = element_text(size = 11, face = "bold"),
              strip.background = element_blank(),
              axis.text = element_text(color="black", size = 10),
              axis.title = element_text(color = "black", size = 16),
              legend.position = "top"
            ) + 
            guides(color = guide_legend(override.aes = list(size = 6)))

    print(p)
    flush.console()
}